# Project 3: Stock Portfolio Tracker
**Data Analyst Portfolio | Python + yfinance**

---

## What this notebook does

1. Loads your portfolio from `portfolio.csv`
2. Fetches the **live current price** of each stock from Yahoo Finance
3. Calculates your **profit or loss** per holding
4. Shows a **1-year price history chart** for each stock
5. Shows a **portfolio breakdown** pie chart

---

## Sample Portfolio

| Ticker | Company | Market | Sector |
|--------|---------|--------|--------|
| TSM | Taiwan Semiconductor | NYSE (US) | Semiconductors |
| GOOG | Alphabet (Google) | NASDAQ (US) | Technology |
| DELTA.BK | Delta Electronics Thailand | SET (TH) | Electronics |
| MINT.BK | Minor International | SET (TH) | Hospitality |
| VT | Vanguard Total World ETF | NYSE (US) | Global ETF |

In [ ]:
# Install required packages — run this cell first!
!pip install yfinance pandas matplotlib --quiet
print('Packages ready!')

## Step 1: Import Libraries

**Libraries** are collections of ready-made code we can reuse. Think of them like Excel add-ins:
- `yfinance` — connects to Yahoo Finance and downloads real stock prices
- `pandas` — organizes data into tables (like a spreadsheet in Python)
- `matplotlib` — draws charts and graphs

In [ ]:
import yfinance as yf            # stock price data from Yahoo Finance
import pandas as pd              # tables and data calculations
import matplotlib.pyplot as plt  # charts and graphs
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded!')

## Step 2: Load Your Portfolio

We use `pandas` to read `portfolio.csv` into Python.
The CSV file contains the stocks you own and the price you paid for each one.

In [ ]:
portfolio = pd.read_csv('portfolio.csv')

print('Portfolio loaded!')
print()
print(portfolio.to_string(index=False))

## Step 3: Fetch Live Prices

We use `yfinance` to get the **current market price** of each stock — real data updated every trading day.

How the function works:
1. Takes a ticker symbol (e.g., `'TSM'`)
2. Downloads the last 5 trading days of data
3. Returns the most recent closing price

> **Note:** If a ticker shows a warning, it means Yahoo Finance had no data for it. Check the ticker spelling.

In [ ]:
def get_current_price(ticker):
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='5d')
        if hist.empty:
            print(f'  Warning: no data for {ticker}')
            return None
        return round(float(hist['Close'].iloc[-1]), 4)
    except Exception as e:
        print(f'  Error fetching {ticker}: {e}')
        return None


print('Fetching live prices from Yahoo Finance...')
portfolio['current_price'] = portfolio['ticker'].apply(get_current_price)

# Check if any tickers failed
failed = portfolio[portfolio['current_price'].isnull()]['ticker'].tolist()
if failed:
    print()
    print(f'  FAILED tickers: {failed}')
    print('  Fix: check the ticker symbols in portfolio.csv and re-run this cell.')
else:
    print()
    print('All prices fetched! Buy price vs. Current price:')
    for _, row in portfolio.iterrows():
        change = row['current_price'] - row['buy_price']
        direction = 'UP  ' if change >= 0 else 'DOWN'
        cur = row['currency']
        ticker = row['ticker']
        buy = row['buy_price']
        now = row['current_price']
        print(f'  {ticker:12} Buy: {cur} {buy:>8,.2f}  Now: {cur} {now:>8,.2f}  [{direction}]')

## Step 4: Calculate Profit & Loss (P&L)

For each holding, we calculate four new columns:

| Column | Formula | Meaning |
|--------|---------|--------|
| `cost_basis` | shares x buy_price | How much you invested |
| `current_value` | shares x current_price | What it is worth today |
| `gain_loss` | current_value - cost_basis | Your profit or loss |
| `gain_loss_pct` | (gain_loss / cost_basis) x 100 | Percentage return |

In [ ]:
# Guard: stop here if any prices are missing
if portfolio['current_price'].isnull().any():
    print('ERROR: Some prices are missing.')
    print('Go back and re-run Step 3 until all prices show OK.')
else:
    portfolio['cost_basis'] = portfolio['shares'] * portfolio['buy_price']
    portfolio['current_value'] = portfolio['shares'] * portfolio['current_price']
    portfolio['gain_loss'] = portfolio['current_value'] - portfolio['cost_basis']
    portfolio['gain_loss_pct'] = (portfolio['gain_loss'] / portfolio['cost_basis'] * 100).round(2)
    print('P&L calculations complete!')

## Step 5: Portfolio Summary

A clean, readable summary of every holding — the kind a financial analyst would present to a manager.

In [ ]:
today = pd.Timestamp.today().strftime('%Y-%m-%d')

print('=' * 65)
print(f'  PORTFOLIO SUMMARY  |  Date: {today}')
print('=' * 65)

for _, row in portfolio.iterrows():
    icon = 'UP  ' if row['gain_loss'] >= 0 else 'DOWN'
    sign = '+' if row['gain_loss'] >= 0 else ''
    cur = row['currency']
    name = row['name']
    ticker = row['ticker']
    shares = int(row['shares'])
    buy = row['buy_price']
    now_price = row['current_price']
    invested = row['cost_basis']
    value = row['current_value']
    gl = row['gain_loss']
    pct = row['gain_loss_pct']

    print()
    print(f'[{icon}]  {name} ({ticker})')
    print(f'       Shares        : {shares}')
    print(f'       Buy Price     : {cur} {buy:>10,.2f}')
    print(f'       Current Price : {cur} {now_price:>10,.2f}')
    print(f'       Invested      : {cur} {invested:>10,.2f}')
    print(f'       Value Now     : {cur} {value:>10,.2f}')
    print(f'       Gain / Loss   : {sign}{cur} {gl:>9,.2f}  ({sign}{pct:.2f}%)')

print()
print('=' * 65)
print('TOTALS BY CURRENCY')
print('=' * 65)

for currency, grp in portfolio.groupby('currency'):
    total_invested = grp['cost_basis'].sum()
    total_value = grp['current_value'].sum()
    total_gl = grp['gain_loss'].sum()
    total_pct = total_gl / total_invested * 100
    sign = '+' if total_gl >= 0 else ''
    print(f'  {currency}: Invested {currency} {total_invested:>10,.2f}  ->  Now {currency} {total_value:>10,.2f}  ({sign}{total_pct:.2f}%)')

## Step 6: Price History Chart (1 Year)

This chart shows how each stock moved over the past 12 months:
- **Green line** = stock is above your buy price (you are in profit)
- **Red line** = stock is below your buy price (you are at a loss)
- **Dashed line** = your buy price (the break-even point)
- **Shading** = profit zone (green) or loss zone (red)

In [ ]:
n = len(portfolio)
fig, axes = plt.subplots(n, 1, figsize=(13, 4 * n))
fig.suptitle('Price History — 1 Year', fontsize=15, fontweight='bold', y=1.01)

if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, portfolio.iterrows()):
    ticker = row['ticker']
    name = row['name']
    cur = row['currency']
    buy = row['buy_price']
    pct = row['gain_loss_pct']

    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='1y')
    except Exception:
        ax.set_title(f'{name} — Failed to fetch data')
        continue

    if hist.empty:
        ax.set_title(f'{name} — No data available')
        continue

    last_price = float(hist['Close'].iloc[-1])
    line_color = 'green' if last_price >= buy else 'red'

    ax.plot(hist.index, hist['Close'], color=line_color, linewidth=1.5, label='Stock Price')
    ax.axhline(y=buy, color='navy', linestyle='--', linewidth=1.2,
               label=f'Buy Price: {cur} {buy:,.2f}')
    ax.fill_between(hist.index, hist['Close'], buy,
                    where=(hist['Close'] >= buy), alpha=0.15, color='green')
    ax.fill_between(hist.index, hist['Close'], buy,
                    where=(hist['Close'] < buy), alpha=0.15, color='red')
    ax.set_title(f'{name}  ({ticker})   {pct:+.2f}%', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'Price ({cur})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('price_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: price_history.png')

## Step 7: Portfolio Breakdown Pie Chart

How is your money distributed across holdings?
A well-diversified portfolio should not have one stock dominating the whole thing.

In [ ]:
currencies = portfolio['currency'].unique()
n_charts = len(currencies)
palette = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63', '#9C27B0']

fig, axes = plt.subplots(1, n_charts, figsize=(7 * n_charts, 6))
fig.suptitle('Portfolio Value Breakdown', fontsize=14, fontweight='bold')

if n_charts == 1:
    axes = [axes]

for ax, currency in zip(axes, currencies):
    grp = portfolio[portfolio['currency'] == currency]
    total = grp['current_value'].sum()
    ax.pie(
        grp['current_value'],
        labels=grp['ticker'],
        autopct='%1.1f%%',
        colors=palette[:len(grp)],
        startangle=140,
        textprops={'fontsize': 11}
    )
    ax.set_title(f'{currency} Holdings  |  Total: {currency} {total:,.2f}', fontsize=12)

plt.tight_layout()
plt.savefig('portfolio_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: portfolio_breakdown.png')

---

## What You Built

Congratulations — this is a **real, working stock portfolio tracker**!

| Skill | How You Used It |
|-------|-----------------||
| **Python** | Wrote code to automate the full analysis |
| **pandas** | Loaded and calculated portfolio data in a table |
| **yfinance** | Fetched live stock prices from Yahoo Finance |
| **matplotlib** | Drew the price history and breakdown charts |

---

> **Portfolio Project 3 — Complete**
>
> Finance background + Python + real market data = a strong signal to any financial employer.